# GPUHub RTX 5090 SW07 Posterior ESS Benchmark

This notebook measures posterior sampling speed for the Python/JAX translation as `seconds_per_min_ess`, using NumPyro NUTS on the bundled `Smets_Wouters_2007_HLT` payload. It does not run Julia on GPUHub. Use the local Julia benchmark artifacts as the CPU reference for likelihood and gradient costs.

Use an RTX 5090 instance with a recent NVIDIA driver. For Blackwell, prefer CUDA 13 JAX wheels. Start with `PROFILE_MODE = "calibration"`; only switch to `PROFILE_MODE = "proper_5090"` after GPU detection, finite gradients, and the Schur support audit look reasonable.

## What This Tests

- GPU visibility and JAX backend/device metadata.
- SW07 fixed-steady-state NumPyro NUTS with a larger safe parameter subset.
- Wall seconds, post-warmup draws/sec, minimum ESS, mean ESS, and seconds per ESS.
- Optional Schur support audit of posterior draws when the HMC run uses `qme_algorithm="doubling"`.
- Contextual comparison to local M4 Julia CPU core likelihood/gradient timings. This is not direct Julia ESS because the Julia harness does not currently run an equivalent NUTS sampler.

In [ ]:
REPO_URL = "https://github.com/matyasfarkas/SurrogateNN_DSGE.git"
BRANCH = "codex/colab-jax-gemini-profile"  # switch to main after merging the benchmark script
ROOT = "/root/gpuhub-tmp/SurrogateNN_DSGE"

# Run order: calibration first, then proper_5090.
PROFILE_MODE = "calibration"  # "calibration" or "proper_5090"

# Precision and solver.
DTYPE = "float32"             # "float32" for speed, "float64" for parity/reference
QME_ALGORITHM = "doubling"    # use Schur support audit when this is doubling
JAX_EXTRA = "auto"            # auto chooses jax[cuda13]>=0.6 or jax[cuda12]>=0.6 from nvidia-smi
FORCE_GPU = True

# Conservative local Julia/M4 reference from benchmarks/results/20260712T083357/combined_results.json.
# Replace after rerunning the Julia CPU benchmark if you want a current same-day reference.
LOCAL_JULIA_REFERENCE = {
    "source": "benchmarks/results/20260712T083357/combined_results.json",
    "sw07_kalman_value_steady_median_s": 0.010629936999999999,
    "sw07_kalman_grad_steady_median_s": 0.0771953125,
    "sw07_switching_value_steady_median_s": 0.207075917,
}


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

def run(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    completed = subprocess.run(cmd, cwd=cwd, check=check, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    return completed

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
run(["nvidia-smi"], check=False)

root = Path(ROOT)
root.parent.mkdir(parents=True, exist_ok=True)
if root.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=root)
    run(["git", "checkout", BRANCH], cwd=root)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=root)
else:
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(root)])

bootstrap_cmd = [
    sys.executable,
    str(root / "scripts" / "gpuhub_bootstrap.py"),
    "--repo-url", REPO_URL,
    "--branch", BRANCH,
    "--root", str(root),
    "--mode", "setup",
    "--jax-extra", JAX_EXTRA,
    "--dtype", DTYPE,
    "--skip-repo-sync",
]
if not FORCE_GPU:
    bootstrap_cmd.append("--allow-cpu")
run(bootstrap_cmd)


In [ ]:
import jax
import jax.numpy as jnp
import numpyro

jax.config.update("jax_enable_x64", DTYPE == "float64")
print("JAX", jax.__version__)
print("NumPyro", numpyro.__version__)
print("x64", jax.config.read("jax_enable_x64"))
print("backend", jax.default_backend())
print("devices", jax.devices())
probe = (jnp.ones((1024, 1024), dtype=jnp.float32) @ jnp.ones((1024, 1024), dtype=jnp.float32)).block_until_ready()
print("matmul dtype", probe.dtype, "sum", float(probe[0, 0]))
if FORCE_GPU and jax.default_backend() != "gpu":
    raise RuntimeError("Expected GPU backend but JAX did not select one.")


In [ ]:
if PROFILE_MODE == "calibration":
    settings = {
        "periods": 80,
        "parameters": "calfa,cg,cgy,crdy,crhob,crpi",
        "warmup": 16,
        "samples": 16,
        "chains": 2,
        "schur_support_draws": 16,
        "preflight": True,
    }
elif PROFILE_MODE == "proper_5090":
    settings = {
        "periods": 160,
        "parameters": "sw07_safe_15",
        "warmup": 256,
        "samples": 256,
        "chains": 4,
        "schur_support_draws": 64,
        "preflight": True,
    }
else:
    raise ValueError(f"Unknown PROFILE_MODE={PROFILE_MODE!r}")

output_path = Path(ROOT) / "benchmarks" / "results" / f"gpuhub_sw07_posterior_ess_{PROFILE_MODE}_{DTYPE}_{QME_ALGORITHM}.json"
cmd = [
    sys.executable,
    "benchmarks/posterior_sampling_speed.py",
    "--preset", "sw07_hlt",
    "--periods", str(settings["periods"]),
    "--parameters", settings["parameters"],
    "--warmup", str(settings["warmup"]),
    "--samples", str(settings["samples"]),
    "--chains", str(settings["chains"]),
    "--chain-method", "vectorized",
    "--dtype", DTYPE,
    "--qme-algorithm", QME_ALGORITHM,
    "--target-accept-prob", "0.8",
    "--max-tree-depth", "8",
    "--prior-width-scale", "0.0025",
    "--prior-width-floor", "0.0001",
    "--schur-support-draws", str(settings["schur_support_draws"]),
    "--output", str(output_path),
]
if FORCE_GPU:
    cmd.append("--force-gpu")
if settings["preflight"]:
    cmd.extend(["--preflight", "--preflight-reps", "1"])

started = time.perf_counter()
run(cmd, cwd=ROOT)
print("Notebook wall seconds", time.perf_counter() - started)
print("Output", output_path)


In [ ]:
result = json.loads(output_path.read_text())
print(json.dumps(result["benchmark"], indent=2))
print(json.dumps(result["throughput"], indent=2))
print(json.dumps(result["extra_fields"], indent=2))
print(json.dumps(result["schur_support_audit"], indent=2))

sec_per_ess = result["throughput"].get("seconds_per_min_ess")
grad_med = None
if result.get("preflight"):
    grad_med = result["preflight"].get("gradient_steady", {}).get("median_s")
print("seconds_per_min_ess", sec_per_ess)
print("gradient_steady_median_s", grad_med)
print("local_julia_reference", json.dumps(LOCAL_JULIA_REFERENCE, indent=2))

audit = result.get("schur_support_audit") or {}
bad = audit.get("doubling_accepts_non_unique_count")
if bad:
    raise RuntimeError(
        f"Doubling accepted {bad} posterior draws that Schur did not classify as unique-stable. "
        "Do not interpret this run as a valid DSGE posterior until support gating is added."
    )


## Interpretation

Use `seconds_per_min_ess` as the headline metric. A lower value is better. `draws_per_second` is secondary because highly autocorrelated chains can produce many draws with little posterior information.

If `schur_support_audit.doubling_accepts_non_unique_count > 0`, the run is a performance experiment only. It is not a valid posterior estimator until the doubling likelihood is gated by Schur determinacy or a GPU-native determinacy certificate is implemented.

For a direct CPU comparison, rerun `benchmarks/profile_validation.py benchmarks/sw07_long_profile.toml` locally after stopping other Julia jobs, then compare Julia's SW07 gradient/likelihood medians with the GPU preflight medians and NumPyro ESS/sec from this notebook.